# 03 — Redes Recurrentes (LSTM + GRU)
Input directo `(N, V_in, 23)` — las capas recurrentes procesan la secuencia temporal.
Catálogo activo: **lstm_s** y **gru_s**. Descomentar `[EXTENDER]` para variantes profundas.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, yfinance as yf
warnings.simplefilter('ignore')

from keras import Sequential, Input
from keras.layers import LSTM, GRU, Dense

from utils import (TICKERS, INPUT_WINDOWS, OUTPUT_WINDOWS,
                   create_time_series_data, make_splits, eval_mae,
                   get_callbacks, compile_model,
                   plot_mae_matrix, build_results_df, best_per_window)

In [ ]:
# ── HIPERPARÁMETROS ───────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 64
QUICK_MODE = False   # True → EPOCHS=20
if QUICK_MODE: EPOCHS = 20

precios = yf.download(TICKERS, start='1945-01-01', auto_adjust=True, progress=False)['Close']
precios.dropna(axis=1, inplace=True)
returns = np.log(precios).diff().dropna()
print(f'Retornos: {returns.shape}  |  EPOCHS={EPOCHS}')

## Catálogo de arquitecturas recurrentes

In [ ]:
# Input shape: (V_in, 23) — NO hace falta aplanar
MODELOS = {
    'lstm_s': lambda V: compile_model(Sequential([
        Input((V, 23)), LSTM(64), Dense(23)])),

    'gru_s':  lambda V: compile_model(Sequential([
        Input((V, 23)), GRU(64),  Dense(23)])),

    # 'lstm_d': lambda V: compile_model(Sequential([          # [EXTENDER] LSTM apilado
    #     Input((V, 23)),
    #     LSTM(64, return_sequences=True), LSTM(32), Dense(23)])),

    # 'gru_d': lambda V: compile_model(Sequential([           # [EXTENDER] GRU apilado
    #     Input((V, 23)),
    #     GRU(64, return_sequences=True),  GRU(32),  Dense(23)])),
}

## Entrenamiento

In [ ]:
results     = {}
historiales = {}

for nombre, build_fn in MODELOS.items():
    for V_in in INPUT_WINDOWS:
        for V_out in OUTPUT_WINDOWS:
            X, y = create_time_series_data(returns, V_in, V_out)
            X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

            model = build_fn(V_in)   # input shape (V_in, 23) directo
            hist  = model.fit(X_tr, y_tr,
                              validation_data=(X_v, y_v),
                              epochs=EPOCHS, batch_size=BATCH_SIZE,
                              callbacks=get_callbacks(), verbose=0)

            key = (nombre, V_in, V_out)
            results[key]     = {'train':  eval_mae(model, X_tr, y_tr),
                                'val':    eval_mae(model, X_v,  y_v),
                                'test':   eval_mae(model, X_ts, y_ts),
                                'params': model.count_params()}
            historiales[key] = hist
            print(f'{nombre}  in={V_in:2d}  out={V_out:2d}  '
                  f'train={results[key]["train"]:.4f}  '
                  f'val={results[key]["val"]:.4f}  '
                  f'test={results[key]["test"]:.4f}  '
                  f'ep={len(hist.history["loss"])}')

## Curvas de convergencia

In [ ]:
for nombre in MODELOS:
    fig, axes = plt.subplots(4, 4, figsize=(16, 12))
    for i, V_in in enumerate(INPUT_WINDOWS):
        for j, V_out in enumerate(OUTPUT_WINDOWS):
            hist = historiales[(nombre, V_in, V_out)]
            ax = axes[i][j]
            ax.plot(hist.history['loss'],     label='train')
            ax.plot(hist.history['val_loss'], label='val')
            ax.set_title(f'in={V_in} out={V_out}', fontsize=8)
            ax.legend(fontsize=6); ax.tick_params(labelsize=6)
    plt.suptitle(f'Convergencia — {nombre}', fontsize=12)
    plt.tight_layout(); plt.show()

## Resultados

In [ ]:
df_res = build_results_df(results)
print(df_res.to_string())

mat = best_per_window(df_res, metric='test')
plot_mae_matrix(mat, title='Recurrentes — mejor MAE test')